# Safer Parks: Capturing perceptions of safety using open data

In [1]:
import pandas as pd
import geopandas as gpd
import fiona
import osmnx as ox
from shapely.ops import unary_union
import re
import folium


In [2]:
# read in safer_parks_functions - see notebook for more details on functions
%run safer_parks_functions.ipynb

### Local Authoirty Approach

This analysis is designed to be run on a local authority by local authority basis as data availability tends to vary by local authority. We use [May 2024 Local Authority District (LAD) data](https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://geoportal.statistics.gov.uk/datasets/ons::local-authority-districts-may-2024-boundaries-uk-bfe-2/about&ved=2ahUKEwi83bT-pJGOAxWBUUEAHSA-NZsQFnoECAoQAQ&usg=AOvVaw3nYML7UR9GdX1gnUYTH8uz), however this can be updated as and when boundaries change. It is also possible to edit the code to change the geographies used by choosing a different shapefile boundary - e.g. focus on an entire unitary authority or metropolitan area, or focus on a specific town or village. 



## Defining park boundaries

Formal park boundary data may exist and be openly avialable for some planing or administrative areas, but often this data is not publicly available or formally defined. We define a process for defining public green space using two different data sources: OS Open Greenspace and Openstreet map. Both sources have their advantages and disadvatages, as outlined in the sections below.

We recommend comparing OS and Openstreetmap data in the UK. Outside the UK we recommend using Openstreet map unless a comparative alternative is available in your region. 



### OS Open Greenspace

In Great Britain the Ordanace Survey provide 'OS Open Greenspace'-  'Covering a range of greenspaces in urban and rural areas including playing fields, sports’ facilities, play areas and allotments.' The data is provided as polygons of the greenspace areas themseleves and entry points to the greensapce site as point data. 

## Openstreetmap Greenspace data


## Comparing Greenspace data 


### Parks in Bradford
In Bradford https://bradforddistrictparks.org/ provides data on formally recognised parks and greenspaces in Bradford. We web-scraped the parks website to get further information on parks faciltiies and define 'formal' aprks and greenspaces within Bradford.




https://osdatahub.os.uk/downloads/open/OpenGreenspace

In [3]:
# read in OS Open Greenspace data
opgrsp_path = "../data_downloads/opgrsp_gpkg_gb/Data/opgrsp_gb.gpkg"

print(fiona.listlayers(opgrsp_path))
os_greenspace_access =gpd.read_file(opgrsp_path, layer ='access_point')
os_greenspace_site =gpd.read_file(opgrsp_path, layer ='greenspace_site')

['access_point', 'greenspace_site']


In [4]:
# read in local authority shapefile 
LAD_path = "../data_downloads/Local_Authority_Districts_May_2024_Boundaries_UK_BFE_-5285665645137523111.geojson"

LAD_2024 =gpd.read_file(LAD_path)

In [5]:
# get OS greenspaces sites in Bradford LAD
greenspace_sites_bradford = subset_to_LAD(LAD_gdf=LAD_2024, LAD_column_name='LAD24NM', LAD_name='Bradford', data_to_subset=os_greenspace_site)
# # combine greenspace sites that intersect or touch into one polygon
# greenspace_sites_bradford = merge_touching_or_intersecting_polygons(greenspace_sites_bradford)
# greenspace_sites_bradford = merge_touching_or_intersecting_polygons(greenspace_sites_bradford)
greenspace_sites_bradford

,id,function,distinctive_name_1,distinctive_name_2,distinctive_name_3,distinctive_name_4,geometry
1,0A0C1723-E2A4-4FA1-B127-F73A7954FB7B,Public Park Or Garden,Shay Grange Crematorium,None,None,None,"MULTIPOLYGON (((413213.450 436356.550, 412835...."
151,2BE88DCC-827A-7FED-E063-AAEFA00A0EDD,Public Park Or Garden,None,None,None,None,"MULTIPOLYGON (((411387.950 437193.900, 411383...."
163,2BE88DCC-83BB-7FED-E063-AAEFA00A0EDD,Play Space,None,None,None,None,"MULTIPOLYGON (((403185.260 444445.820, 403177...."
605,2BE88DCC-D52D-7FED-E063-AAEFA00A0EDD,Other Sports Facility,None,None,None,None,"MULTIPOLYGON (((417825.010 434672.350, 417828...."
754,2BE88DCC-F082-7FED-E063-AAEFA00A0EDD,Play Space,Attock Park,None,None,None,"MULTIPOLYGON (((418208.790 433169.630, 418211...."
...,...,...,...,...,...,...,...
161497,2BE88F5F-A26E-7FED-E063-AAEFA00A0EDD,Playing Field,None,None,None,None,"MULTIPOLYGON (((411672.850 437052.390, 411676...."
161534,2BE88F5F-AA93-7FED-E063-AAEFA00A0EDD,Other Sports Facility,None,None,None,None,"MULTIPOLYGON (((413680.210 431865.180, 413682...."
161759,2BE88F5F-D91C-7FED-E063-AAEFA00A0EDD,Public Park Or Garden,None,None,None,None,"MULTIPOLYGON (((418047.600 430223.370, 418052...."
161827,2BE88F5F-E527-7FED-E063-AAEFA00A0EDD,Public Park Or Garden,None,None,None,None,"MULTIPOLYGON (((414592.290 428702.530, 414583...."


In [6]:
# get OS greenspaces access points in Bradford LAD
greenspace_access_bradford = subset_to_LAD(LAD_gdf=LAD_2024, LAD_column_name='LAD24NM', LAD_name='Bradford', data_to_subset=os_greenspace_access)
greenspace_access_bradford.explore()

In [7]:
##### Uncomment code to define GS using OSM instead od the OS Open Greenspace data, 
###### replace place name or polygon with area of interest 


# get OSM greenspaces
## Either using a placename or an shapefile to define the spatial limits of the data

# Define the tags for i.e. parks, woodlands, and scrublands
tags = {
    'leisure': ['park'],
    'landuse': ['forest', 'meadow'],
    'natural': ['wood', 'scrub']
}
################################################################################
#### Get the geometries for the specified place name and tags ####

# # Define the place name
# place_name = "Bradford, West Yorkshire, England"
# # get OSM geometries from palcename
# gdf = ox.geometries_from_place(place_name, tags)

############################ OR ################################################
#### Get the geometries for the specified polygon and tags ####

# define LAD of interest e.g. Bradford
bradford = LAD_2024.loc[LAD_2024['LAD24NM']=='Bradford',:]
# get OSM geometries from polygon boundary 
gdf = ox.geometries_from_polygon(bradford.to_crs("EPSG:4326").unary_union, tags)

################################################################################
# # Filter the geometries to include only parks, woodlands, and scrublands
osm_greenspace= gdf[gdf['leisure'].isin(['park'])|
                             gdf['landuse'].isin(['forest', 'meadow']) | 
                             gdf['natural'].isin(['wood', 'scrub'])].reset_index()

#osm_greenspace = merge_touching_or_intersecting_polygons(osm_greenspace)
osm_greenspace 

/tmp/ipykernel_1071/499009049.py:28: FutureWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in the v2.0.0 release. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  gdf = ox.geometries_from_polygon(bradford.to_crs("EPSG:4326").unary_union, tags)


,element_type,osmid,access,barrier,geometry,source,name,addr:city,addr:housenumber,addr:postcode,...,source:alt_name,landcover,addr:housename,addr:village,source:geometry,historic:ref:1835_tithe_award,sport,fixme,ways,type
0,way,4472848,NaN,NaN,"POLYGON ((-1.74235 53.91227, -1.74210 53.91220...",NaN,Burley Village Green,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,way,4483806,NaN,NaN,"POLYGON ((-1.74985 53.91496, -1.74974 53.91512...",NaN,Victoria Park,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,way,4535372,NaN,NaN,"POLYGON ((-1.81990 53.92424, -1.81964 53.92424...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,way,4535557,NaN,NaN,"POLYGON ((-1.82648 53.92484, -1.82659 53.92463...",NaN,Canker Well,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,way,4536859,NaN,NaN,"POLYGON ((-1.82357 53.92632, -1.82360 53.92627...",interpolation;Bing,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3149,relation,16458862,NaN,NaN,"POLYGON ((-1.78064 53.92493, -1.78056 53.92494...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[91889679, 1214074937]",multipolygon
3150,relation,16493384,NaN,NaN,"POLYGON ((-1.75957 53.92112, -1.75960 53.92115...",Bing,Jerry Holme,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[988606691, 272999006]",multipolygon
3151,relation,16618634,NaN,NaN,"POLYGON ((-1.75531 53.89982, -1.75465 53.90013...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[1219327737, 1219327738]",multipolygon
3152,relation,17373549,NaN,NaN,"POLYGON ((-1.76340 53.86633, -1.76274 53.86657...",NaN,Hawksworth Spring,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[1160819168, 1265257477]",multipolygon


In [8]:
# Get OS greenspace cols to join
os_gs_to_join =greenspace_sites_bradford[['distinctive_name_1','function','id','geometry']]
os_gs_to_join.columns = ['Name (OS)', 'Type (OS)', 'OS ID', 'geometry' ]


# Get OSM greenspace cols to join
# combine OSM tags into one value
osm_greenspace['Type (OSM)'] =osm_greenspace[['natural', 'leisure', 'landuse']].apply(lambda x: ', '.join(sorted(x.dropna().astype(str))),axis=1)

osm_gs_to_join = osm_greenspace[['name','Type (OSM)','osmid','geometry']]
osm_gs_to_join.columns = ['Name (OSM)', 'Type (OSM)', 'OSM ID', 'geometry' ]

# change crs before joining
osm_gs_to_join= osm_gs_to_join.to_crs(os_gs_to_join.crs)
# append OSM data to OS data
os_osm_greenspace_bradford =pd.concat([os_gs_to_join,osm_gs_to_join])

# unify greenspace geographies, keepign type data
os_osm_greenspace_bradford =merge_touching_or_intersecting_polygons_condense(os_osm_greenspace_bradford)

# remove duplicate information in variables, generated during process of combining into one geometry
os_osm_greenspace_bradford['Name (OS)'] =os_osm_greenspace_bradford['Name (OS)'].apply(lambda x: ', '.join(sorted(set(map(str.strip, str(x).split(','))))) if pd.notnull(x) else x).apply(lambda x: re.sub(r'(?<!\w),', '', x))
os_osm_greenspace_bradford['Name (OSM)'] =os_osm_greenspace_bradford['Name (OSM)'].apply(lambda x: ', '.join(sorted(set(map(str.strip, str(x).split(','))))) if pd.notnull(x) else x).apply(lambda x: re.sub(r'(?<!\w),', '', x))

os_osm_greenspace_bradford['Type (OS)'] =os_osm_greenspace_bradford['Type (OS)'].apply(lambda x: ', '.join(sorted(set(map(str.strip, str(x).split(', '))))) if pd.notnull(x) else x).apply(lambda x: re.sub(r'(?<!\w),', '', x))
os_osm_greenspace_bradford['Type (OSM)'] =os_osm_greenspace_bradford['Type (OSM)'].apply(lambda x: ', '.join(sorted(set(map(str.strip, str(x).split(', '))))) if pd.notnull(x) else x).apply(lambda x: re.sub(r'(?<!\w),', '', x))

os_osm_greenspace_bradford['OS ID'] = os_osm_greenspace_bradford['OS ID'].apply(lambda x: ', '.join(sorted(set(map(str.strip, str(x).split(', '))))) if pd.notnull(x) else x).apply(lambda x: re.sub(r'(?<!\w),', '', x))
os_osm_greenspace_bradford['OSM ID'] = os_osm_greenspace_bradford['OSM ID'].apply(lambda x: ', '.join(sorted(set(map(str.strip, str(x).split(', '))))) if pd.notnull(x) else x).apply(lambda x: re.sub(r'(?<!\w),', '', x))

In [9]:
os_osm_greenspace_bradford.explore()

/home/maeve/miniforge3/envs/parks-dev-env-old/lib/python3.12/site-packages/folium/features.py:1204: UserWarning: GeoJsonTooltip is not configured to render for GeoJson GeometryCollection geometries. Please consider reworking these features: [{'Name (OS)': '', 'Type (OS)': ' ', 'OS ID': ' ', 'Name (OSM)': ' Cringles Plantation, Little Sea Moor', 'Type (OSM)': 'meadow, wood', 'OSM ID': '1334814971.0, 1354777437.0, 1355127608.0, 1355127613.0, 1355127624.0, 1355127626.0, 1355127630.0, 1355127648.0, 1355127654.0, 1355127658.0, 1355127660.0, 1359967022.0, 1359968748.0, 1359968752.0, 1359968754.0, 1359979811.0, 1360767488.0, 1360767493.0, 1360767497.0, 1360767499.0, 1360767505.0, 1360767508.0, 1360767512.0, 1360767523.0, 1360767528.0, 1360769053.0, 1360769055.0, 1360769056.0, 1360769057.0, 1360769060.0, 1360769061.0, 1360769064.0, 1360769065.0, 1360769066.0, 1360769067.0, 1360769069.0, 1361072023.0, 1361072024.0, 1361072026.0, 1361072028.0, 1361072032.0, 1361072036.0, 1361072038.0, 1361072048

In [10]:
# Defining parks

bradford_parks_data = '../data_downloads/bradford_district_parks.geojson'

# Data scraped from: 'https://bradforddistrictparks.org/park/page/'
# See Bradford_parks_info.ipynb
bd_parks= gpd.read_file(bradford_parks_data)
bd_parks.explore()

In [11]:
# os_osm_greenspace_bradford_buffer = os_osm_greenspace_bradford
# os_osm_greenspace_bradford_buffer['geometry'] =os_osm_greenspace_bradford_buffer.buffer(50)


# test =gpd.sjoin(os_osm_greenspace_bradford_buffer, bd_parks.to_crs(os_osm_greenspace_bradford_buffer.crs), predicate='intersects', how='left')

# test.loc[test['Park Name'].isna()==True, 'BMDC_park']= 'No'
# test.loc[test['Park Name'].isna()==False, 'BMDC_park']= 'BMDC Park'

In [12]:
##### combine bradford parks data from website with OSM and OS combined greensapce data
# cerate a copy
bd_parks_polygon= bd_parks
# remove geom column (for now)
bd_parks_polygon =bd_parks_polygon.drop(columns=['geometry'])
bd_parks_polygon = bd_parks_polygon.reset_index()

# idenitfy where bradford park anmes exactly match the OS or OSM Name
bd_parks_polygon_1 =bd_parks_polygon.merge(os_osm_greenspace_bradford, right_on='Name (OS)',left_on='Park Name', how='left')
bd_parks_polygon_2 =bd_parks_polygon.merge(os_osm_greenspace_bradford, right_on='Name (OSM)',left_on='Park Name', how='left')

# remove null rows after join
bd_parks_polygon_1 =bd_parks_polygon_1.dropna(subset=['OS ID','OSM ID'])
bd_parks_polygon_2 =bd_parks_polygon_2.dropna(subset=['OS ID','OSM ID'])

# combine the exact match data into one geodataframe
bd_parks_polygon_1_2 =gpd.GeoDataFrame(pd.concat([bd_parks_polygon_1,bd_parks_polygon_2]).drop_duplicates('index'))

# use the exact match data to identify the remiaingn bmdc parks without os or osm greenspace data
bd_parks_polygon_3 =bd_parks_polygon[~bd_parks_polygon['index'].isin(bd_parks_polygon_1_2['index'])]
# and save as bd_parks_polygon_3
bd_parks_polygon_3 =gpd.GeoDataFrame(bd_parks_polygon_3, geometry=gpd.points_from_xy(bd_parks_polygon_3.Longitude, bd_parks_polygon_3.Latitude, crs="EPSG:4326"))

# ## Uncomment if want to do sjoin on buffer in future
# os_osm_greenspace_bradford_buffer = os_osm_greenspace_bradford
# os_osm_greenspace_bradford_buffer['geometry'] =os_osm_greenspace_bradford.buffer(30)

# spaially join bmdc park points to os and OSM greenspace data
bd_parks_polygon_3 =gpd.sjoin(bd_parks_polygon_3.to_crs(os_osm_greenspace_bradford.crs),os_osm_greenspace_bradford, predicate='intersects', how='left')

###### come back to later- bmdc parks on website that do not have an easily identifiable open counterpart #####
#(either due to lack of spatial info on bmdc, innacurate locaiton or incosnsitent naming)
bd_parks_polygon_4 =bd_parks_polygon_3[bd_parks_polygon_3['index_right'].isna()]


bd_parks_polygon_3=bd_parks_polygon_3[~bd_parks_polygon_3['index_right'].isna()]
# define bradford parks as those with an exact name or spatial join match (43/71 parks, a remaining 10 aprks do have long/lat info)
bradford_parks_gdf =gpd.GeoDataFrame(pd.concat([bd_parks_polygon_1,bd_parks_polygon_2,bd_parks_polygon_3]).drop_duplicates('index'))




In [13]:
bradford_parks_gdf

,index,Park Name,Park URL,Location,Opening Hours,Latitude,Longitude,Car parking,River,Walking routes,...,Miniature railway,Park features str,Name (OS),Type (OS),OS ID,Name (OSM),Type (OSM),OSM ID,geometry,index_right
5,5,Russell Hall Park,https://bradforddistrictparks.org/park/russell...,"Russell Hall Ave, Queensbury, BD13 2AL","Open 24 hours a day, all year round",53.7652724,-1.8488896,0,0,0,...,0,"5-a-side football pitch, Bowling green, Childr...",Russell Hall Park,"Bowling Green, Play Space, Playing Field","2BE88E26-152D-7FED-E063-AAEFA00A0EDD, 2BE88E26...",Russell Hall Park,park,200908882.0,"POLYGON ((575797.160 5957983.962, 575793.517 5...",NaN
6,6,Greenwood Park,https://bradforddistrictparks.org/park/greenwo...,"Wilsden Road, Allerton, BD15 9AD","Open 24 hours a day, all year round",53.814044,-1.8366054,0,0,0,...,0,"Basketball court, Childrens play area, Cricket...",Greenwood Park,"Other Sports Facility, Play Space, Playing Field","2BE88DD4-0458-7FED-E063-AAEFA00A0EDD, 2BE88E25...",Greenwood Park,park,42862661.0,"POLYGON ((576722.801 5963482.189, 576665.305 5...",NaN
7,7,Grange Park,https://bradforddistrictparks.org/park/grange-...,"Main Street, Burley-in-Wharfedale, Ilkley, LS2...","Open 24 hours a day, all year round",53.9130668,-1.7492053,0,0,0,...,0,"Childrens play area, Fitness equipment",Grange Park,"Bowling Green, Play Space, Public Park Or Garden","2BE88E23-F4D0-7FED-E063-AAEFA00A0EDD, 2BE88E23...",Grange Park,park,60671740.0,"POLYGON ((582106.481 5974534.031, 582106.473 5...",NaN
8,8,Foxhill Park,https://bradforddistrictparks.org/park/foxhill...,"Fleet Lane, Queensbury, BD13 2JR","Open 24 hours a day, all year round",53.7712639,-1.8547914,0,0,0,...,0,Full size football pitch,Foxhill Park,Playing Field,2BE88E26-0BD1-7FED-E063-AAEFA00A0EDD,Foxhill Park,park,168515340.0,"POLYGON ((575314.442 5958697.058, 575314.649 5...",NaN
14,14,Knowles Park,https://bradforddistrictparks.org/park/knowles...,"Holme Wood, Bradford, BD4 9PP","Open 24 hours a day, all year round",53.7724228,-1.7194417,1,0,1,...,0,"Car parking, Childrens play area, Dog free are...",Knowles Park,"Bowling Green, Play Space, Public Park Or Garden","2BE88DD9-604B-7FED-E063-AAEFA00A0EDD, 2BE88E28...",Knowles Park,park,54444001.0,"POLYGON ((584420.492 5958868.434, 584417.832 5...",NaN
23,23,Seymour Street Recreation Ground,https://bradforddistrictparks.org/park/seymour...,"Upper Seymour Street, \r\nBradford,\r\nBD3 9LJ",NA,None,None,0,0,0,...,0,NA,Seymour Street Recreation Ground,"Other Sports Facility, Play Space, Playing Field","2BE88E27-85A2-7FED-E063-AAEFA00A0EDD, 2BE88E27...",Seymour Park,park,27359839.0,"POLYGON ((583390.488 5961242.250, 583386.479 5...",NaN
26,26,Eccleshill Park,https://bradforddistrictparks.org/park/ecclesh...,"Rimswell Holt, Greengates, BD10 0EY","Open 24 hours a day, all year round",53.8242976,-1.7105699,0,0,0,...,0,"BMX track, Childrens play area, Fitness equipm...",Eccleshill Park,"Play Space, Public Park Or Garden","2BE88E27-B71A-7FED-E063-AAEFA00A0EDD, 2BE88E27...",,wood,"1240758815.0, 1240758816.0","MULTIPOLYGON (((584890.460 5964678.190, 584884...",NaN
28,28,Cross Roads Park,https://bradforddistrictparks.org/park/cross-r...,NA,NA,None,None,0,0,0,...,0,NA,Cross Roads Park,"Bowling Green, Other Sports Facility, Play Spa...","2BE88E23-74A7-7FED-E063-AAEFA00A0EDD, 2BE88E23...",Cross Roads Park,park,1081617151.0,"POLYGON ((570290.516 5965934.824, 570300.733 5...",NaN
40,40,Wibsey Park,https://bradforddistrictparks.org/park/wibsey-...,"Wibsey Park Avenue, Bradford BD6 3QD","Open 24 hours a day, all year round",53.76673410000001,-1.787095700000009,0,0,1,...,0,"BMX track, Bowling green, Childrens play area,...",Wibsey Park,"Bowling Green, Other Sports Facility, Play Spa...","2BE88DD3-BEA9-7FED-E063-AAEFA00A0EDD, 2BE88DE0...",Wibsey Park,"park, wood","200726700.0, 200947589.0, 958222047.0","POLYGON ((579687.387 5958070.287, 579688.126 5...",NaN
49,49,Prince of Wales Park,https://bradforddistrictparks.org/park/prince-...,"Park Drive, 

In [14]:
#### NEED TO BUG SOLVE ONLY SAVING POLYGON_3 as points. 
bd_parks_polygon_3.explore()

In [15]:

# get boundsing box of 400m arround each park to calcualte network connectivity 
bradford_parks_gdf =bradford_parks_gdf.to_crs('EPSG:2770')
# conver back to 4326 after buffer for bbox crs
cols =bradford_parks_gdf.buffer(400).to_crs('EPSG:4326').bounds.add_prefix('400m_bounding_').columns
bradford_parks_gdf[cols]= bradford_parks_gdf.buffer(400).to_crs('EPSG:4326').bounds.add_prefix('400m_bounding_')
bradford_parks_gdf

,index,Park Name,Park URL,Location,Opening Hours,Latitude,Longitude,Car parking,River,Walking routes,...,OS ID,Name (OSM),Type (OSM),OSM ID,geometry,index_right,400m_bounding_minx,400m_bounding_miny,400m_bounding_maxx,400m_bounding_maxy
5,5,Russell Hall Park,https://bradforddistrictparks.org/park/russell...,"Russell Hall Ave, Queensbury, BD13 2AL","Open 24 hours a day, all year round",53.7652724,-1.8488896,0,0,0,...,"2BE88E26-152D-7FED-E063-AAEFA00A0EDD, 2BE88E26...",Russell Hall Park,park,200908882.0,"POLYGON ((8448768.006 6998427.250, 8448764.841...",NaN,-1.855965,53.760832,-1.841952,53.769118
6,6,Greenwood Park,https://bradforddistrictparks.org/park/greenwo...,"Wilsden Road, Allerton, BD15 9AD","Open 24 hours a day, all year round",53.814044,-1.8366054,0,0,0,...,"2BE88DD4-0458-7FED-E063-AAEFA00A0EDD, 2BE88E25...",Greenwood Park,park,42862661.0,"POLYGON ((8443867.032 7001767.852, 8443823.797...",NaN,-1.841258,53.810252,-1.827201,53.818701
7,7,Grange Park,https://bradforddistrictparks.org/park/grange-...,"Main Street, Burley-in-Wharfedale, Ilkley, LS2...","Open 24 hours a day, all year round",53.9130668,-1.7492053,0,0,0,...,"2BE88E23-F4D0-7FED-E063-AAEFA00A0EDD, 2BE88E23...",Grange Park,park,60671740.0,"POLYGON ((8435581.160 7011893.989, 8435581.464...",NaN,-1.756059,53.909031,-1.742106,53.917074
8,8,Foxhill Park,https://bradforddistrictparks.org/park/foxhill...,"Fleet Lane, Queensbury, BD13 2JR","Open 24 hours a day, all year round",53.7712639,-1.8547914,0,0,0,...,2BE88E26-0BD1-7FED-E063-AAEFA00A0EDD,Foxhill Park,park,168515340.0,"POLYGON ((8447864.339 6998278.046, 8447863.869...",NaN,-1.862943,53.766891,-1.847155,53.775736
14,14,Knowles Park,https://bradforddistrictparks.org/park/knowles...,"Holme Wood, Bradford, BD4 9PP","Open 24 hours a day, all year round",53.7724228,-1.7194417,1,0,1,...,"2BE88DD9-604B-7FED-E063-AAEFA00A0EDD, 2BE88E28...",Knowles Park,park,54444001.0,"POLYGON ((8451751.492 7007151.197, 8451752.392...",NaN,-1.728002,53.767632,-1.710852,53.777120
23,23,Seymour Street Recreation Ground,https://bradforddistrictparks.org/park/seymour...,"Upper Seymour Street, \r\nBradford,\r\nBD3 9LJ",NA,None,None,0,0,0,...,"2BE88E27-85A2-7FED-E063-AAEFA00A0EDD, 2BE88E27...",Seymour Park,park,27359839.0,"POLYGON ((8448999.693 7007213.359, 8449028.205...",NaN,-1.741298,53.789419,-1.727552,53.797631
26,26,Eccleshill Park,https://bradforddistrictparks.org/park/ecclesh...,"Rimswell Holt, Greengates, BD10 0EY","Open 24 hours a day, all year round",53.8242976,-1.7105699,0,0,0,...,"2BE88E27-B71A-7FED-E063-AAEFA00A0EDD, 2BE88E27...",,wood,"1240758815.0, 1240758816.0","MULTIPOLYGON (((8446348.162 7010193.080, 84463...",NaN,-1.717435,53.819734,-1.703269,53.828980
28,28,Cross Roads Park,https://bradforddistrictparks.org/park/cross-r...,NA,NA,None,None,0,0,0,...,"2BE88E23-74A7-7FED-E063-AAEFA00A0EDD, 2BE88E23...",Cross Roads Park,park,1081617151.0,"POLYGON ((8438634.851 6996641.521, 8438666.735...",NaN,-1.939164,53.831962,-1.925510,53.840558
40,40,Wibsey Park,https://bradforddistrictparks.org/park/wibsey-...,"Wibsey Park Avenue, Bradford BD6 3QD","Open 24 hours a day, all year round",53.76673410000001,-1.787095700000009,0,0,1,...,"2BE88DD3-BEA9-7FED-E063-AAEFA00A0EDD, 2BE88DE0...",Wibsey Park,"park, wood","200726700.0, 200947589.0, 958222047.0","POLYGON ((8450415.405 7002223.882, 8450410.844...",NaN,-1.796840,53.761323,-1.777691,53.771574
49,49,Prince of Wales Park,https://bradforddistrictparks.org/park/prince-...,"Park Drive, Bingley, BD16 3DF","Open 24 hours a day, all year round",53.85672470000001,-1.8258287999999538,0,0,1,...,2BE88E23-BC1A-7FED-E063-AAEFA00A0EDD,Prince of Wales Park,park,180572882.0,"POLYGON ((8439382.271 7004544.312, 8439566.401...",NaN,-1.834680,53.852313,-1.818216,53.862332


In [22]:
bradford_parks_gdf.to_file("Data/bradford_parks_geom.geojson", driver="GeoJSON")

In [16]:
import osmnx as ox

In [17]:
# G = ox.graph_from_bbox(bradford_parks_gdf.iloc[0]['400m_bounding_maxy'],
#                         bradford_parks_gdf.iloc[0]['400m_bounding_miny'],
#                         bradford_parks_gdf.iloc[0]['400m_bounding_maxx'],
#                         bradford_parks_gdf.iloc[0]['400m_bounding_minx'],
#                         network_type='walk')

# # Calculate the number of corners (nodes with degree 3 or more)
# corners = [node for node, degree in dict(G.degree()).items() if degree >= 3]
# num_corners = len(corners)

# # Calculate the connectiveness (average node degree)
# avg_node_degree = sum(dict(G.degree()).values()) / len(G.nodes)

# print(f"Number of corners: {num_corners}")
# print(f"Connectiveness (average node degree): {avg_node_degree:.2f}")


# # Convert the network into a GeoDataFrame
# gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)

# g

In [19]:
# calculate n corners and connectedness of each network arround each park in the gdf 
bradford_parks_gdf['avg_node_degree_400m_network']=''
bradford_parks_gdf['n_corners_400m_nework']=''
for i in range(0,bradford_parks_gdf.shape[0]):  
    bbox=(
        bradford_parks_gdf.iloc[i]['400m_bounding_maxy'],
        bradford_parks_gdf.iloc[i]['400m_bounding_miny'],
        bradford_parks_gdf.iloc[i]['400m_bounding_maxx'],
        bradford_parks_gdf.iloc[i]['400m_bounding_minx']
        )  
    G = ox.graph_from_bbox(bbox=bbox,
                            network_type='walk')

    # Calculate the number of corners (nodes with degree 3 or more)
    corners = [node for node, degree in dict(G.degree()).items() if degree >= 3]
    num_corners = len(corners)

    # Calculate the connectiveness (average node degree)
    avg_node_degree = sum(dict(G.degree()).values()) / len(G.nodes)
    
    # bradford_parks_gdf.loc[bradford_parks_gdf.index[i], 'avg_node_degree_400m_network'] = avg_node_degree.astype(float)
    bradford_parks_gdf.loc[bradford_parks_gdf.index[i], 'avg_node_degree_400m_network'] = avg_node_degree
    bradford_parks_gdf.loc[bradford_parks_gdf.index[i], 'n_corners_400m_network'] = num_corners


    print(f"Number of corners: {num_corners}")
    print(f"Connectiveness (average node degree): {avg_node_degree:.2f}")


    # Convert the network into a GeoDataFrame
    #gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)

/tmp/ipykernel_1071/3182730832.py:11: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = ox.graph_from_bbox(bbox=bbox,


Number of corners: 210
Connectiveness (average node degree): 4.71


/tmp/ipykernel_1071/3182730832.py:11: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = ox.graph_from_bbox(bbox=bbox,


Number of corners: 82
Connectiveness (average node degree): 4.68


/tmp/ipykernel_1071/3182730832.py:11: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = ox.graph_from_bbox(bbox=bbox,


...

Number of corners: 174
Connectiveness (average node degree): 4.96


/tmp/ipykernel_1071/3182730832.py:11: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = ox.graph_from_bbox(bbox=bbox,


Number of corners: 7
Connectiveness (average node degree): 4.00


/tmp/ipykernel_1071/3182730832.py:11: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = ox.graph_from_bbox(bbox=bbox,


Number of corners: 153
Connectiveness (average node degree): 5.12


In [20]:
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)

In [21]:
gdf_nodes.to_file("Data/gdf_nodes.geojson", driver="GeoJSON")
gdf_edges.to_file("Data/gdf_edges.geojson", driver="GeoJSON")

osmid  oneway           name  \
u           v           key                                                  
21154171    304918610   0                  27891306   False     Sunny Bank   
            1038682591  0    [701123738, 701123740]   False     Otley Road   
21154172    530965904   0                 175770488   False  Bradford Road   
            7101279799  0                 701124760   False     Otley Road   
            520827289   0                 175770490   False  Bradford Road   
...                                             ...     ...            ...   
10857547687 530965889   0                  42526755   False            NaN   
            2279776972  0                 218754456   False            NaN   
11868890017 10857547666 0                1167492567   False            NaN   
            567653109   0                  44750289   False            NaN   
            10857547676 0                1167492567   False            NaN   

                                 highway reversed   length  \
u           v           key                                  
21154171    304918610   0    residential     True   95.749   
            1038682591  0        primary     True   69.519   
21154172    530965904   0          trunk    False   16.376   
            7101279799  0        primary    False   36.781   
            520827289   0          trunk     True   88.519   
...                                  ...      ...      ...   
10857547687 530965889   0        footway     True  176.863   
            2279776972  0        footway    False  108.311   
11868890017 10857547666 0        footway     True    7.716   
            567653109   0        footway     True   88.367   
            10857547676 0        footway    False   81.193   

                                                                      geometry  \
u           v           key                                                      
21154171    304918610   0    LINESTRING (-1.77695 53.83014, -1.77714 53.830...   
            1038682591  0    LINESTRING (-1.77695 53.83014, -1.77716 53.829...   
21154172    530965904   0    LINESTRING (-1.78015 53.82773, -1.78009 53.82758)   
            7101279799  0    LINESTRING (-1.78015 53.82773, -1.77980 53.82798)   
            520827289   0    LINESTRING (-1.78015 53.82773, -1.78043 53.828...   
...                                                                        ...   
10857547687 530965889   0    LINESTRING (-1.78749 53.82770, -1.78734 53.827...   
            2279776972  0    LINESTRING (-1.78749 53.82770, -1.78748 53.828...   
11868890017 10857547666 0    LINESTRING (-1.78672 53.82915, -1.78663 53.82919)   
            567653109   0    LINESTRING (-1.78672 53.82915, -1.78713 53.829...   
            10857547676 0    LINESTRING (-1.78672 53.82915, -1.78693 53.829...   

                              lanes    ref maxspeed bridge  
u           v           key                                 
21154171    304918610   0       NaN    NaN      NaN    NaN  
            1038682591  0    [3, 2]  A6038   30 mph    NaN  
21154172    530965904   0       NaN   A650   30 mph    NaN  
            7101279799  0         3  A6038   30 mph    NaN  
            520827289   0       NaN   A650   30 mph    NaN  
...                             ...    ...      ...    ...  
10857547687 530965889   0       NaN    NaN      NaN    NaN  
            2279776972  0       NaN    NaN      NaN    NaN  
11868890017 10857547666 0       NaN    NaN      NaN    NaN  
            567653109   0       NaN    NaN      NaN    NaN  
            10857547676 0       NaN    NaN      NaN    NaN  

[492 rows x 11 columns]